Generate rubrics

In [ ]:
# ✅ log on Hugging Face（used to load model）
import os
from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"))

/home/aychen/miniconda3/envs/rubrics/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM

# set up the model
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")
gen_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.17it/s]
Device set to use cuda:0


In [ ]:
import os, json

def build_prompt_med(conversation_path, reference_path, fewshot_path="data/few_shot.jsonl", lang="zh"):
    with open(conversation_path, "r", encoding="utf-8") as f:
        target_conversation = f.read().strip()

    if os.path.exists(reference_path):
        with open(reference_path, "r", encoding="utf-8") as f:
            reference = f.read().strip()
        if len(reference) > 800:
            reference = reference[:800] + "…（Truncated）"
        reference_info = f"【Reference Info（Optional. Only supplementary. Do not deviate from the user's questions.）】\n{reference}\n\n"
    else:
        reference_info = "【Reference Info（Optional）】\n\n"

    fewshot_block = ""
    if fewshot_path and os.path.exists(fewshot_path):
        try:
            with open(fewshot_path, "r", encoding="utf-8") as f:
                fewshots = [json.loads(line) for line in f.readlines()]
            fs_parts = []
            for i, ex in enumerate(fewshots[:1]):  
                prob = ex.get("problem", "")
                rubs = ex.get("checklist", [])
                rub_texts = [f"- {r}" for r in rubs if isinstance(r, str) and r.strip()]
                fs_parts.append(
                    f"【Example {i+1}】\nproblem:{prob}\nrubrics：\n" + "\n".join(rub_texts)
                )
            fewshot_block = "\n\n".join(fs_parts) + "\n\n"
        except Exception as e:
            print("⚠️ few-shot parse error:", e)
            fewshot_block = ""

    # —— Key prompt words (Chinese, 4, no axis/point) --
    prompt = (
        "You are a medical evaluation assistant. Your task is to generate rubrics (evaluation criteria) for assessing model responses in medical conversations.\n"
        "You will be given EXAMPLES of how to generate rubrics. Then, you will be asked to generate rubrics for a NEW conversation.\n\n"
        "Each rubric must:\n"
        "- Be a clear, actionable evaluation criterion.\n"
        "- Directly relate to the given conversation and its medical content.\n"
        " Rubrics should be checklist items, used to verify whether the responses cover key medical information.\n"
        " Each rubric item must be a specific knowledge point or checklist item related to the medical content. Rather than the general accuracy/completeness/communication items.\n"
        " If you find the provided reference materials helpful, please align the key points and expressions therein. If you determine that the reference information is irrelevant, please ignore it. Generate the rubrics/checklist items on your own and based on your medical knowledge.\n"
        "=== FEW-SHOT EXAMPLES ===\n"
        f"{fewshot_block}"
        "\n=== TARGET CONVERSATION ===\n"
        f"{target_conversation}\n\n"
        f"{reference_info}"
        "Please ensure the following when generating rubrics:\n\n"
        "- Generate **4 distinct criteria in CHINESE** for each conversation. \n"
        "- Each rubric must be **directly related to the specific conversation content**. Do not include generic or unrelated criteria.\n"
        "Now generate rubrics in JSON format as a list. Please only output the JSON array; The length of the array is fixed at 4. Each element is a string, and each one is a rubric, avoiding numbering and redundant explanations.\n"
        "Rubrics:\n"
    )
    return prompt


In [ ]:
import re, json

def extract_rubrics_med(output_text):
    # Prioritize capturing the JSON array
    m = re.search(r"\[\s*(\".*?\"(\s*,\s*\".*?\")*)\s*\]", output_text, flags=re.S)
    if m:
        try:
            arr = json.loads(m.group(0))
            # Keep only the strings, cut to 4 or fill in the blanks
            arr = [s.strip() for s in arr if isinstance(s, str)]
            return (arr[:4] + [""]*4)[:4]
        except Exception:
            pass
    # fallback
    lines = [l.strip("-• \t") for l in output_text.strip().splitlines() if l.strip()]
    lines = [l for l in lines if len(l) > 3][:4]
    return (lines + [""]*4)[:4]


In [ ]:
def generate_rubrics_med(conversation_id,
                         conversation_dir="outputs/prompts",
                         reference_dir="outputs/rag",
                         out_dir="outputs/rubrics_llmeval_med",
                         fewshot_path="data/few_shot.jsonl"):
    os.makedirs(out_dir, exist_ok=True)
    conversation_path = os.path.join(conversation_dir, f"conversation_{conversation_id}.txt")
    reference_path    = os.path.join(reference_dir,    f"reference_{conversation_id}.txt")
    output_path       = os.path.join(out_dir,          f"rubrics_{conversation_id}.json")

    prompt = build_prompt_med(conversation_path, reference_path, fewshot_path=fewshot_path)
    
    print("\n📮 Prompt Preview:\n")
    print(prompt)

    # Generate
    output = gen_pipeline(
        prompt,
        max_length=7000,
        max_new_tokens=1500,
        do_sample=True,
        temperature=0.7
    )[0]["generated_text"]

    rubrics = extract_rubrics_med(output)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(rubrics, f, ensure_ascii=False, indent=2)
        
    print(f"✅ Saved rubrics to {output_path}")
    print("\n🔍 Conversation:\n")
    print(open(conversation_path).read())
    print("📌 Rubrics Preview:", json.dumps(rubrics, ensure_ascii=False))
    return rubrics


In [54]:
# run for conversation_0.txt + reference_0.txt
generate_rubrics_med("13", fewshot_path="data/few_shot.jsonl")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=6000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📮 Prompt Preview:

You are a medical evaluation assistant. Your task is to generate rubrics (evaluation criteria) for assessing model responses in medical conversations.
You will be given EXAMPLES of how to generate rubrics. Then, you will be asked to generate rubrics for a NEW conversation.

Each rubric must:
- Be a clear, actionable evaluation criterion.
- Directly relate to the given conversation and its medical content.
 Rubrics should be checklist items, used to verify whether the responses cover key medical information.
 Each rubric item must be a specific knowledge point or checklist item related to the medical content. Rather than the general accuracy/completeness/communication items.
 If you find the provided reference materials helpful, please align the key points and expressions therein. If you determine that the reference information is irrelevant, please ignore it. Generate the rubrics/checklist items on your own and based on your medical knowledge.
=== FEW-SHOT EXAMPLES 

✅ Saved rubrics to outputs/rubrics_llmeval_med/rubrics_13.json

🔍 Conversation:

User: 得过水痘又接种水痘疫苗，身体会有什么反应吗？

📌 Rubrics Preview: ["解释水痘疫苗的预防作用，特别是对儿童的保护作用。", "描述水痘的传染性，包括病毒的传染性时期和传播途径。", "解释水痘的症状和体征，包括皮疹的三个阶段和可能的伴发症状。", "强调水痘疫苗的安全性和有效性，特别是对预防水痘和其伴发的健康问题。"]


['解释水痘疫苗的预防作用，特别是对儿童的保护作用。',
 '描述水痘的传染性，包括病毒的传染性时期和传播途径。',
 '解释水痘的症状和体征，包括皮疹的三个阶段和可能的伴发症状。',
 '强调水痘疫苗的安全性和有效性，特别是对预防水痘和其伴发的健康问题。']

In [ ]:
from glob import glob

def batch_generate_llmeval_med(start=0, end=None, fewshot_path="data/few_shot.jsonl"):
    convo_files = sorted(glob("outputs/prompts/conversation_*.txt"))
    ids = [os.path.splitext(os.path.basename(p))[0].split("_")[1] for p in convo_files]
    ids = sorted(ids, key=lambda x: int(x))  
    if end is None:
        end = len(ids)
    for cid in ids[start:end]:
        try:
            generate_rubrics_med(cid, fewshot_path=fewshot_path)
        except Exception as e:
            print(f"❌ rubrics failed for {cid}: {e}")


In [ ]:
batch_generate_llmeval_med(start=13)

In [ ]:
import os
import json
from glob import glob

def find_missing_rubrics(conversation_dir="outputs/prompts",
                         rubrics_dir="outputs/rubrics_llmeval_med"):
    # First, find all the conversation files (benchmarks)
    convo_files = sorted(glob(os.path.join(conversation_dir, "conversation_*.txt")))
    all_ids = [int(os.path.splitext(os.path.basename(f))[0].split("_")[1]) for f in convo_files]

    # Then find the existing rubrics files
    rubrics_files = sorted(glob(os.path.join(rubrics_dir, "rubrics_*.json")))
    done_ids = [int(os.path.splitext(os.path.basename(f))[0].split("_")[1]) for f in rubrics_files]

    # missing
    missing_ids = sorted(set(all_ids) - set(done_ids))

    print(f"🔍 Total conversations: {len(all_ids)}")
    print(f"✅ Rubrics already generated: {len(done_ids)}")
    print(f"❌ Missing rubrics: {len(missing_ids)}")
    print("缺少的 index:", missing_ids)

    return missing_ids

# usage：
missing_ids = find_missing_rubrics()



🔍 Total conversations: 114
✅ Rubrics already generated: 94
❌ Missing rubrics: 20
缺少的 index: [29, 37, 42, 51, 55, 58, 65, 66, 67, 69, 74, 79, 86, 87, 91, 95, 96, 108, 110, 112]


: 

In [106]:
# run for conversation_0.txt + reference_0.txt
generate_rubrics_med("37", fewshot_path="data/few_shot.jsonl")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=1500) and `max_length`(=7000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📮 Prompt Preview:

You are a medical evaluation assistant. Your task is to generate rubrics (evaluation criteria) for assessing model responses in medical conversations.
You will be given EXAMPLES of how to generate rubrics. Then, you will be asked to generate rubrics for a NEW conversation.

Each rubric must:
- Be a clear, actionable evaluation criterion.
- Directly relate to the given conversation and its medical content.
 Rubrics should be checklist items, used to verify whether the responses cover key medical information.
 Each rubric item must be a specific knowledge point or checklist item related to the medical content. Rather than the general accuracy/completeness/communication items.
 If you find the provided reference materials helpful, please align the key points and expressions therein. If you determine that the reference information is irrelevant, please ignore it. Generate the rubrics/checklist items on your own and based on your medical knowledge.
=== FEW-SHOT EXAMPLES 

✅ Saved rubrics to outputs/rubrics_llmeval_med/rubrics_37.json

🔍 Conversation:

User: 做阑尾切除术一般多长时间能完全恢复好？有没有需要注意的？

📌 Rubrics Preview: ["rubrics content (string)", "rubrics content (string)", "rubrics content (string)", "rubrics content (string)"]


['rubrics content (string)',
 'rubrics content (string)',
 'rubrics content (string)',
 'rubrics content (string)']

In [ ]:
# Regenerate the missing rubrics
for cid in missing_ids:
    try:
        generate_rubrics_med(str(cid), fewshot_path="data/few_shot.jsonl")
    except Exception as e:
        print(f"⚠️ Failed for {cid}: {e}")